# 14 — Kelanjutan bertahap, review → subset → detektor → evaluasi
Jalankan SATU tahap tiap sesi. Upload paket kode dan dataset asli/assets. Attach output kernel tahap sebelumnya (13 untuk select) sebagai input; review p100 yang sudah diisi diunggah sebagai dataset kecil berisi review_p100.csv. Jangan mengunggah token.
Tahap default `select` menggunakan CPU, berhenti bila review belum lengkap. Tidak ada auto-accept atau auto-run 45 detektor.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, zipfile, subprocess
WORK = Path('/kaggle/working/texture_study_v1')
INPUT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=False)
# Kaggle unpacks an uploaded zip into plain files; accept either form, never both.
archives = list(INPUT.rglob('texture_study_code.zip'))
# A previous stage's kernel output also carries a code copy next to study/; only the code dataset counts.
unpacked = [p.parent for p in INPUT.rglob('package_manifest.json')
            if (p.parent/'code/cyclegan/tools/texture_study.py').exists() and not (p.parent/'study').exists()]
if len(archives) + len(unpacked) != 1:
    raise ValueError(f'Attach exactly one texture study code dataset; zip={archives}, unpacked={unpacked}')
if archives:
    with zipfile.ZipFile(archives[0]) as z:
        for item in z.infolist():
            if not (WORK/item.filename).resolve().is_relative_to(WORK.resolve()): raise ValueError('unsafe zip')
        z.extractall(WORK)
else:
    shutil.copytree(unpacked[0], WORK, dirs_exist_ok=True)
    for p in [WORK, *WORK.rglob('*')]: p.chmod(p.stat().st_mode | 0o200)  # /kaggle/input is read-only
import hashlib
manifest = json.loads((WORK/'package_manifest.json').read_text())
bad = [k for k,v in manifest.items() if hashlib.sha256((WORK/k).read_bytes()).hexdigest() != v]
if bad: raise ValueError(f'Package files differ from package_manifest.json: {bad[:5]}')
print('code package verified:', len(manifest), 'files')
HERE = WORK/'code/cyclegan'
ENV = {**os.environ, 'PYTHONPATH':str(HERE), 'OMP_NUM_THREADS':'2', 'MKL_NUM_THREADS':'2', 'PYTHONUNBUFFERED':'1'}
def unique(marker):
    hits=sorted({p.parent for p in INPUT.rglob(marker)})
    if len(hits)!=1: raise ValueError(f'Expected one {marker}; found {hits}')
    return hits[0]
DATASET = unique('manifest.csv')
ASSETS = unique('reviewed_masks.json')
ENV.update(MELON_DATASET=str(DATASET),MELON_ASSETS=str(ASSETS))
os.environ.update(ENV)
def run(*args):
    subprocess.run([sys.executable,*map(str,args)],cwd=HERE,env=ENV,check=True)
CLI = HERE/'tools/texture_study.py'
import torch
print(torch.__version__,torch.cuda.is_available())


In [ ]:
STAGE='select'  # select | subset-generators | build-detectors | train-detectors | evaluate | plot
FRACTION=25    # 25/50; 100 tersedia dari notebook13
source=unique('study.json')   # attach the previous stage's kernel output (13 for select)
STUDY=WORK/'study'
shutil.copytree(source,STUDY)
# Filled blind review arrives as a small dataset file, not as a re-uploaded study snapshot.
overrides=list(INPUT.rglob('review_p100.csv'))
if len(overrides)>1: raise ValueError(f'Attach at most one review_p100.csv; found {overrides}')
if overrides:
    import csv, hashlib
    target=STUDY/'p100/review/review.csv'
    ids=lambda path:[r['blind_id'] for r in csv.DictReader(path.open(newline=''))]
    if ids(overrides[0])!=ids(target): raise ValueError('review_p100.csv blind IDs/order differ from the study')
    target.chmod(target.stat().st_mode|0o200); shutil.copyfile(overrides[0],target)
    print('p100 review override sha256', hashlib.sha256(target.read_bytes()).hexdigest())
for p in [STUDY,*STUDY.rglob('*')]: p.chmod(p.stat().st_mode|0o200)  # copied from read-only input
if STAGE=='select':
    run(CLI,'select','--study',STUDY)
    print((STUDY/'selection.json').read_text())
elif STAGE=='subset-generators':
    if not torch.cuda.is_available():raise RuntimeError('GPU required')
    winner=json.loads((STUDY/'selection.json').read_text())['winner']
    for stage in ['train','export','diagnose','review']:
        args=[CLI,stage,'--study',STUDY,'--fraction',FRACTION,'--arms','G0',winner]
        if stage=='train':args+=['--budget-minutes',30]
        if stage=='export':args+=['--dataset',DATASET,'--device','cuda']
        if stage=='diagnose':args+=['--device','cuda']
        run(*args)
    print('STOP: review this subset before building detector arms')
elif STAGE=='build-detectors':
    run(CLI,'build-detectors','--study',STUDY,'--fraction',FRACTION,'--dataset',DATASET)
elif STAGE=='train-detectors':
    if not torch.cuda.is_available():raise RuntimeError('GPU required')
    run(CLI,'train-detectors','--study',STUDY,'--fraction',FRACTION)
elif STAGE=='evaluate':
    run(CLI,'evaluate','--study',STUDY,'--fraction',FRACTION,'--device','cpu','--bootstrap',1000)
elif STAGE=='plot':
    run(CLI,'plot','--study',STUDY)
else:raise ValueError('unknown stage')


## Interpretasi
Detektor: lima arm × tiga seed × tiga subset =45 run, diluncurkan per subset sesudah reservasi GPU.
Seluruh arm memakai 3000 update, batch8,640px,YOLOv8n pretrained, validation tiap250. Evaluator memuat checkpoint terpilih, bukan otomatis bobot final.
Bootstrap berpasangan memakai kelompok foto; variasi seed dilaporkan terpisah. Kalibrasi threshold dan pengukuran recall memisahkan kelompok validation, tetapi checkpoint tetap dipilih pada validation penuh; hasil eksploratif.
Generator satu seed,20epoch; tidak menjanjikan peningkatan AP. Test lama tidak dibaca. SKConv/TC-UNet bukan bagian run awal.
